# Transparent forecast-based decision support (Phase 14)
The finalized model predicts Appliances energy in **Wh, 10 minutes ahead**. Four deterministic rules translate forecast magnitude into conservative recommendations. This is decision support, not automatic building or equipment control: no controllable system, intervention data, or causal action model is available. No savings are estimated or guaranteed.

## Development calibration
Only TRAIN + VALIDATION targets (`target_appliances_next`) calibrate the linear 50th, 75th, 90th and 95th percentiles. TEST is never opened for calibration, examples or rule design. The exploratory whole-dataset threshold is not used. Input checksums tie calibration to the verified Phase 6 splits.

In [1]:
from pathlib import Path
import sys
import json
root = Path.cwd().resolve()
while not (root / "AGENTS.md").is_file():
    if root == root.parent:
        raise RuntimeError("Run within the project")
    root = root.parent
sys.path.insert(0, str(root))
from ml.src import recommendation_engine as engine
from ml.src import model_inference as inference
thresholds = engine.load_thresholds()
assert thresholds == engine.calibrate_thresholds()
thresholds

Out[1]: 
{'calibration_dataset': 'train+validation',
 'calibration_input_sha256': {'data/processed/energy_train.csv': 'b4e74536ad61bce5a75b044066f9012348a12cbd69095a0e5a801642fc7ebbe4',
  'data/processed/energy_validation.csv': '5b93ccdb622696544eb3abb7d59b59445bf37c5ff6dfe2f8449b4b1282e41533'},
 'calibration_rows': 15917,
 'forecast_horizon_minutes': 10,
 'level_definitions': {'ELEVATED': 'percentile_75 <= prediction < percentile_90',
  'HIGH': 'percentile_90 <= prediction < percentile_95',
  'NORMAL': 'prediction < percentile_75',
  'VERY HIGH': 'prediction >= percentile_95'},
 'percentile_50': 60.0,
 'percentile_75': 100.0,
 'percentile_90': 190.0,
 'percentile_95': 320.0,
 'percentile_method': 'numpy.percentile(method=linear)',
 'schema_version': 1,
 'target_column': 'target_appliances_next',
 'test_usage_statement': 'TEST was not opened or used for threshold calibration or rule design.',
 'test_used': False,
 'unit': 'Wh'}


## Rules and trend
NORMAL is below p75; ELEVATED includes p75 and excludes p90; HIGH includes p90 and excludes p95; VERY HIGH includes p95. p50 is descriptive only. Recommendations review flexible/non-critical loads only where safe and operationally feasible. Essential equipment must remain safe.

Trend is the exact sign of predicted minus current Wh, with no tolerance: stable means exact equality. Percentage change exists only for positive current energy. Trend does not change the recommendation level and does not imply sensor causation.

In [2]:
engine.LEVELS, {level: rule[0] for level, rule in engine.RULES.items()}

Out[2]: 
({'NORMAL': 'prediction < percentile_75',
  'ELEVATED': 'percentile_75 <= prediction < percentile_90',
  'HIGH': 'percentile_90 <= prediction < percentile_95',
  'VERY HIGH': 'prediction >= percentile_95'},
 {'NORMAL': 'NORMAL_CONSUMPTION',
  'ELEVATED': 'ELEVATED_CONSUMPTION',
  'HIGH': 'HIGH_CONSUMPTION',
  'VERY HIGH': 'VERY_HIGH_CONSUMPTION'})


## Deterministic real development examples
Predict every development observation in chronological order using the persisted model. Show the **first forecast in each band**, including normal and high consumption. This selection uses forecast level only, never prediction error or TEST. Development examples illustrate behavior, not generalization performance. Model fitting and tuning are not performed.

In [3]:
development = engine.development_data()
bundle = inference.load_final_model()
features = development[bundle[1]["ordered_feature_names"]]
results = engine.predict_and_recommend(features, model_bundle=bundle, thresholds=thresholds)
examples = {}
for position, result in enumerate(results):
    level = result["prediction"]["level"]
    if level not in examples:
        examples[level] = {"origin": development.date.iloc[position].isoformat(), "result": result}
assert set(examples) == set(engine.LEVELS)
for level in engine.LEVELS:
    print(level)
    print(json.dumps(examples[level], indent=2, allow_nan=False))

NORMAL
{
  "origin": "2016-01-18T17:00:00",
  "result": {
    "prediction": {
      "value_wh": 55.505409240722656,
      "horizon_minutes": 10,
      "unit": "Wh",
      "level": "NORMAL"
    },
    "thresholds_wh": {
      "percentile_50": 60.0,
      "percentile_75": 100.0,
      "percentile_90": 190.0,
      "percentile_95": 320.0
    },
    "trend": {
      "current_wh": 40.0,
      "change_wh": 15.505409240722656,
      "change_percent": 38.76352310180664,
      "direction": "rising"
    },
    "recommendation": {
      "code": "NORMAL_CONSUMPTION",
      "priority": "low",
      "title": "No high-consumption warning",
      "message": "No immediate high-consumption warning is detected for the upcoming 10-minute period.",
      "rationale": "Forecast 55.5054 Wh satisfies prediction < percentile_75 using TRAIN + VALIDATION target percentiles; thresholds are in thresholds_wh."
    },
    "scope": {
      "automatic_control": false,
      "estimated_savings": null,
      "note": "Fo

In [4]:
first = engine.predict_and_recommend(features.iloc[:1], model_bundle=bundle, thresholds=thresholds)
assert first == engine.predict_and_recommend(features.iloc[:1], model_bundle=bundle, thresholds=thresholds)
assert all(r["scope"]["automatic_control"] is False and r["scope"]["estimated_savings"] is None for r in results)
assert inference.sha256(inference.ARTIFACT_PATH) == bundle[1]["model_artifact_sha256"]
print("Deterministic inference, Wh/10-minute contract, unchanged model and decision-support scope verified.")

Deterministic inference, Wh/10-minute contract, unchanged model and decision-support scope verified.


## Limitations and later integration
Percentile bands are descriptive development-distribution rules, not validated intervention effectiveness or alert calibration. Discrete energy readings can create equal thresholds and empty bands. Distribution drift and forecast error can affect usefulness. Negative forecasts are rejected, not clipped. No occupancy, equipment identity, comfort evidence, confidence intervals, or causal sensor effects are inferred. No energy, cost or CO2 savings are fabricated.

`predict_and_recommend` returns a JSON-safe list with one result per row, preserving positional order, including single-row input. Later API work can expose this structure; Phase 14 adds no backend or frontend. Callers remain responsible for valid origin-time sensor readings and historical features.